# Unit 4: Training with Weak Targets: Basic

[Course overview](00_overview.ipynb)

<ul>
<li><a href="#learn" style="color:#E65100;"><strong>Overview and Learning Objectives</strong></a></li>
<li><a href="#unfolding" style="color:black;">Unfolded Weak Targets</a></li>
<li><a href="#batching" style="color:#1B5E20;">Background: Variable-Length Targets and Batching</a></li>
<li><a href="#local_cost" style="color:black;">SDTW with Local BCE</a></li>
<li><a href="#modified_targets" style="color:#1B5E20;">Background: Modified Targets</a></li>
<li><a href="#scheduling" style="color:#1B5E20;">Background: Temperature Scheduling</a></li>
<li><a href="#weak_training" style="color:black;">Training with Unfolded Targets</a></li>
<li><a href="#predictions" style="color:black;">Inspecting Predictions</a></li>
<li><a href="#exercise_unfolding" style="color:#006064;">Exercise 1: Understanding Unfolding</a></li>
</ul>

<a id="learn"></a>

<div style="
  background-color: #FFE0B2A0;
  border: 1px solid #FFC671A0;
  padding-top: 10px;
  padding-bottom: 10px;
  padding-left: 20px;
  padding-right: 20px;
  margin-top: 10px;
  margin-bottom: 10px;
">

<h2 style="color:#E65100; margin-top:0;">Overview and Learning Objectives</h2>

<p style="color:#E65100;">We now train from weak targets who are in the correct order, but where the target durations are unknown. In this first experiment, we uniformly unfold the targets and use SDTW with
    unit step weights. Afterwards, we discuss valid sequence lengths, modified targets, and
    temperature scheduling.</p>

</div>

In [ ]:
from time import perf_counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from torch.utils.data import DataLoader
from ddtw import SDTW

from dataset_multipitch import MultipitchDataset, collate_multipitch
from model import Transcription_CNN

# --- Figure display and style ---
%config InlineBackend.figure_formats = ['svg']
plt.rcParams.update({"font.size": 8, "axes.titlesize": 9})

In [ ]:
# config
torch.manual_seed(1)
device = torch.device("cpu")

max_polyphony = 3  # maximum number of simultaneous onsets in the dataset
# activation function for the output layer of the model
output_activation = "sigmoid"
freqs = (30, 35, 40, 45)  # frequencies (Hz) of the sinusoids in the dataset

In [ ]:
# data
train_data = MultipitchDataset(
    size=256, seed=0, noise_std=0.01, max_polyphony=max_polyphony, freqs=freqs
)
valid_data = MultipitchDataset(
    size=32, seed=1, noise_std=0.01, max_polyphony=max_polyphony, freqs=freqs
)
train_loader = DataLoader(
    train_data,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_multipitch,
    num_workers=0,
)
valid_loader = DataLoader(
    valid_data, batch_size=16, collate_fn=collate_multipitch, num_workers=0
)
class_names = train_data.class_names

<a id="scenario"></a>

## Weak targets

In the previous notebook, we discussed strong targets with shape `(500, 4)` that
specify frequency activity at each frame.
Weak targets retain the ordered state sequence and discard its durations by
removing consecutive duplicate vectors. 

In [ ]:
# plot a sample waveform and its corresponding strong and weak targets

(
    test_waveform,
    test_strong_targets_frames,
    test_weak_targets_frames,
    _,
    _,
) = valid_data[0]

fig, ax = plt.subplots(3, 1, figsize=(6.2, 4.8))

im0 = ax[0].plot(np.arange(0, 500, 1), test_waveform[0].numpy(), color="k")
ax[0].set(
    xlim=[0, 500],
    title="Input waveform",
    ylabel="Amplitude",
    xlabel="Frame index",
)

im1 = ax[1].imshow(
    test_strong_targets_frames.numpy().T,
    aspect="auto",
    origin="lower",
    interpolation="None",
    cmap="gray_r",
    extent=[0, 500, -0.5, len(class_names) - 0.5],
)
ax[1].set(
    xlim=[0, 500],
    title="Strong targets (frames)",
    ylabel="Class",
    yticks=np.arange(len(class_names)),
    yticklabels=class_names,
    xlabel="Frame index",
)
plt.colorbar(im1, ax=ax[1], label="Amplitude")
cbar = plt.colorbar(im1, ax=ax[0], label=None)
fig.delaxes(cbar.ax)

im2 = ax[2].imshow(
    test_weak_targets_frames.numpy().T,
    aspect="auto",
    origin="lower",
    interpolation="None",
    cmap="gray_r",
)
ax[2].set(
    title="Weak targets (frames)",
    ylabel="Class",
    yticks=np.arange(len(class_names)),
    yticklabels=class_names,
    xticks=np.arange(0, test_weak_targets_frames.shape[0], 1),
    xlabel="Weak target index",
)
plt.colorbar(im2, ax=ax[2], label="Amplitude")

fig.tight_layout()
plt.show()

<a id="exercise_targets"></a>

<div style="
  background-color: #B2EBF290;
  border: 1px solid #4DD0E190;
  padding-top: 10px;
  padding-bottom: 10px;
  padding-left: 20px;
  padding-right: 20px;
  margin-top: 10px;
  margin-bottom: 10px;
">

<strong style="color:#006064; display:block; break-after:avoid; page-break-after:avoid;">Exercise 1: From Strong to Weak Targets</strong>

<ol style="color:#006064;">
<li>Remove consecutive duplicate rows from the strong targets. Retain a state if it reappears after a different state.</li>
<li>Check equality with the dataset's weak targets.</li>
</ol>

</div>

### Solution: From Strong to Weak Targets

Compare adjacent rows across all frequency channels. Always keep the first
row and then retain a row whenever any channel changes. This discards
durations while preserving the sequence, including repeated nonadjacent
states and silence.

In [ ]:
strong = test_strong_targets_frames
keep = torch.cat(
    [
        torch.tensor([True]),
        torch.any(strong[1:] != strong[:-1], dim=1),
    ]
)
weak_from_strong = strong[keep]
torch.testing.assert_close(weak_from_strong, test_weak_targets_frames)
print("Strong / weak lengths:", len(strong), len(weak_from_strong))

<a id="duration_assumption"></a>

<div style="
  background-color: #C8E6C9A0;
  border: 1px solid #81C784A0;
  padding: 10px 20px;
  margin: 10px 0;
">

<h2 style="color:#1B5E20; margin-top:0;">Background: The Duration Assumption in Unfolding</h2>

If a sequence contains $M$ compact states and $N=500$ prediction frames,
unfolding gives each state about $N/M$ target positions. This introduces
approximate timing information through the representation. In the presence
of uniform step weights, paths along the diagonal are favorable. This can stabilize
learning initially, but equal state durations need not match the waveform. The SDTW
alignment remains necessary to correct these approximate boundaries.

</div>

<a id="local_cost"></a>

## SDTW with Local BCE


The local cost compares prediction frame $n$ with target position $m$:

$$C(n,m)=-\sum_k[y_{m,k}\log x_{n,k}
+(1-y_{m,k})\log(1-x_{n,k})].$$

`cost_function="BCE"` sums over frequency classes. `normalization="N"`
divides the sequence SDTW score by the prediction length before averaging
over the batch. Strong targets are used for validation metrics, but never enter
the weak-target training loss.

<a id="scheduling"></a>

<div style="
  background-color: #C8E6C9A0;
  border: 1px solid #81C784A0;
  padding: 10px 20px;
  margin: 10px 0;
">

<h2 style="color:#1B5E20; margin-top:0;">Background: Softmin Temperature Scheduling</h2>

A larger initial gamma distributes weight across more paths when early
predictions provide little alignment information. Reducing gamma later
sharpens paths and correspondences. This changes the objective itself: the bound
$\min_\mathbf{A} c_\mathbf{A}-\gamma\log K\leq L_{\rm SDTW}
\leq\min_\mathbf{A} c_\mathbf{A}$ depends on gamma and the path set. Jumps in the loss after
temperature updates need not mean that the prediction quality deteriorated.
Compare the alignment, modified targets, and frame F1 as well.

</div>

<a id="weak_training"></a>

## Training with Unfolded Targets


In [ ]:
# initialize SDTW

sdtw_loss = SDTW(
    gamma=10.0,
    step_sizes=[[1, 0], [1, 1], [0, 1]],
    cost_function="BCE",
    global_step_weights=[1.0, 1.0, 1.0],
    normalization="N",
    backend="cpu_numba",
)

# initialize model
model_sdtw_unfolded = Transcription_CNN(
    num_classes=train_data.num_classes, activation=output_activation
).to(device)

optimizer = torch.optim.Adam(model_sdtw_unfolded.parameters(), lr=0.001)
train_losses, valid_losses, precisions, recalls, fmeasures = [], [], [], [], []
training_start = perf_counter()

for epoch in range(100):
    # train step
    model_sdtw_unfolded.train()
    total_loss = 0.0
    for (
        waveforms,
        strong_targets_frames,
        weak_targets_frames,
        strong_targets_onsets,
        weak_targets_onsets,
        target_lengths_frames,
        target_lengths_onsets,
    ) in train_loader:
        optimizer.zero_grad()
        predictions = model_sdtw_unfolded(waveforms)

        # compute SDTW loss between predictions and unaligned weak targets
        weak_targets_frames_unfolded = torch.stack(
            [
                weak_targets_frames[
                    b,
                    np.linspace(
                        0,
                        target_lengths_frames[b],
                        500,
                        endpoint=False,
                        dtype=int,
                    ),
                    :,
                ]
                for b in range(weak_targets_frames.shape[0])
            ],
            dim=0,
        )

        loss = sdtw_loss(
            X=predictions, Y=weak_targets_frames_unfolded
        )  # , list_M=target_lengths_frames)

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(waveforms)
    train_losses.append(total_loss / len(train_data))

    # validation step
    model_sdtw_unfolded.eval()
    total_loss = 0.0
    with torch.no_grad():
        p_ = []
        r_ = []
        f_ = []
        for (
            waveforms,
            strong_targets_frames,
            weak_targets_frames,
            strong_targets_onsets,
            weak_targets_onsets,
            target_lengths_frames,
            target_lengths_onsets,
        ) in valid_loader:
            predictions = model_sdtw_unfolded(waveforms)

            weak_targets_frames_unfolded = torch.stack(
                [
                    weak_targets_frames[
                        b,
                        np.linspace(
                            0,
                            target_lengths_frames[b],
                            500,
                            endpoint=False,
                            dtype=int,
                        ),
                        :,
                    ]
                    for b in range(weak_targets_frames.shape[0])
                ],
                dim=0,
            )
            loss = sdtw_loss(
                X=predictions, Y=weak_targets_frames_unfolded
            )  # , list_M=target_lengths_frames)

            total_loss += loss.item() * len(waveforms)

            # calculate precision, recall, and F1 score for each batch
            pred_thresh = (predictions > 0.5) * 1.0
            TP = (strong_targets_frames * pred_thresh).sum()
            FP = ((1 - strong_targets_frames) * pred_thresh).sum()
            FN = (strong_targets_frames * (1 - pred_thresh)).sum()
            P = TP / (TP + FP)
            R = TP / (TP + FN)
            F = 2 * P * R / (P + R)
            p_.append(P.item())
            r_.append(R.item())
            f_.append(F.item())

    valid_losses.append(total_loss / len(valid_data))

    # print stats every 10 epochs
    elapsed = perf_counter() - training_start
    if epoch == 0 or (epoch + 1) % 10 == 0:
        print(
            (
                "Epoch %3d | train loss %.5f | val loss %.4f | "
                "val P %.4f | val R %.4f | val F1 %.4f | "
                "elapsed time %.1f s"
            )
            % (
                epoch + 1,
                train_losses[-1],
                valid_losses[-1],
                np.mean(p_),
                np.mean(r_),
                np.mean(f_),
                elapsed,
            )
        )
        precisions.append(np.mean(p_))
        recalls.append(np.mean(r_))
        fmeasures.append(np.mean(f_))

    # do a gamma annealing every 20 epochs
    if (epoch + 1) % 20 == 0:
        sdtw_loss.gamma *= 0.5
        print("Gamma updated to %.2f" % sdtw_loss.gamma)

# plot training and validation loss curves
fig, ax = plt.subplots(1, 2, figsize=(6.2, 2.2))
ax[0].plot(np.arange(1, len(train_losses) + 1), train_losses, label="train")
ax[0].plot(
    np.arange(1, len(valid_losses) + 1), valid_losses, label="validation"
)
ax[0].set(
    xlabel="Epoch", ylabel="Mean loss", title="Training and validation loss"
)
ax[0].legend()
ax[0].grid()

ax[1].plot(np.arange(1, len(precisions) + 1), precisions, label="precision")
ax[1].plot(np.arange(1, len(recalls) + 1), recalls, label="recall")
ax[1].plot(np.arange(1, len(fmeasures) + 1), fmeasures, label="F1 score")
ax[1].set(
    xlabel="Epoch", ylabel="Score", title="Precision, recall, and F1 score"
)
ax[1].legend()
ax[1].grid()
plt.show()

<a id="predictions"></a>

## Inspecting Predictions


We now compare the strong reference, predicted probabilities, thresholded errors, and
individual channel responses. Black denotes TP, red FP, cyan FN, and white TN.

In [ ]:
(
    test_waveform,
    test_strong_targets_frames,
    test_weak_targets_frames,
    _,
    _,
) = valid_data[0]
pred_weak_sdtw_unfolded = model_sdtw_unfolded(test_waveform[None])[0]

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(5.2, 5), squeeze=False)

ax[0, 0].imshow(
    test_strong_targets_frames.numpy().T,
    aspect="auto",
    origin="lower",
    interpolation="None",
    cmap="gray_r",
    clim=[0, 1],
)
ax[1, 0].imshow(
    pred_weak_sdtw_unfolded.detach().numpy().T,
    aspect="auto",
    origin="lower",
    interpolation="None",
    cmap="gray_r",
    clim=[0, 1],
)

# thresholded predictions
pred_weak_thresh = (pred_weak_sdtw_unfolded > 0.5) * 1.0
TP = test_strong_targets_frames * pred_weak_thresh
FP = (1 - test_strong_targets_frames) * pred_weak_thresh
FN = test_strong_targets_frames * (1 - pred_weak_thresh)
TN = (1 - test_strong_targets_frames) * (1 - pred_weak_thresh)
pred_RGB_sdtw = np.zeros((500, len(class_names), 3))
pred_RGB_sdtw[TP == 1] = [0, 0, 0]
pred_RGB_sdtw[FP == 1] = [1, 0, 0]
pred_RGB_sdtw[FN == 1] = [0, 1, 1]
pred_RGB_sdtw[TN == 1] = [1, 1, 1]
ax[2, 0].imshow(
    pred_RGB_sdtw.swapaxes(0, 1), aspect="auto", interpolation="None"
)

for d in range(len(class_names)):
    ax[3, 0].plot(np.arange(500), np.zeros(500) + d, color="grey", linewidth=2)
    ax[3, 0].plot(
        np.arange(500),
        pred_weak_sdtw_unfolded.detach().numpy()[:, d] + d,
        color=f"C{d}",
        label=class_names[d],
        linewidth=1,
    )
    ax[3, 0].plot(
        np.arange(500),
        test_strong_targets_frames[:, d] + d,
        color=f"C{d}",
        linestyle="--",
    )
    ax[3, 0].grid()
ax[3, 0].plot(np.arange(500), np.zeros(500) + d + 1, color="grey", linewidth=2)
ax[3, 0].set(xlim=[-0.5, 499.5])


ax[1, 0].set(ylabel="Probabilities")
ax[2, 0].set(ylabel="thresh. pred.")
ax[3, 0].set(ylabel="signals", xlabel="Frame index")

ax[0, 0].set(title="Unfolded weak targets and SDTW", ylabel="Strong targets")
ax[3, 0].set(xlabel="Frame index")

red_patch = mpatches.Patch(color="red", label="FP")
cyan_patch = mpatches.Patch(color="cyan", label="FN")
black_patch = mpatches.Patch(color="black", label="TP")

ax[2, 0].legend(
    handles=[black_patch, red_patch, cyan_patch], loc="upper right"
)

for a_ in ax.flatten():
    a_.set(yticks=np.arange(len(class_names)), yticklabels=class_names)

fig.tight_layout()
plt.show()

<a id="exercise_unfolding"></a>

<div style="
  background-color: #B2EBF290;
  border: 1px solid #4DD0E190;
  padding-top: 10px;
  padding-bottom: 10px;
  padding-left: 20px;
  padding-right: 20px;
  margin-top: 10px;
  margin-bottom: 10px;
">

<strong style="color:#006064; display:block; break-after:avoid; page-break-after:avoid;">Exercise 1: Why Does Target Unfolding Help?</strong>

<ol style="color:#006064;">
<li>Explain which duration assumptions unfolding introduces, and compare its predictions with the strong-target baseline and compact-target experiment in Unit 5.</li>
<li>Treat the unfolded weak targets as strong targets and train with a frame-wise BCE. Analyze the raw and thresholded predictions, and compare them to the previous SDTW training run.</li>
<li>Extend the unfolded-SDTW training diagnostics to plot soft alignments and modified targets for validation samples at the first epoch and every ten epochs. Explain their evolution.</li>
</ol>

</div>

### Solution: Why Does Target Unfolding Help?

1. Each state receives roughly equal duration. This supplies a diagonal
   prior which, however, can be wrong when state durations vary greatly.
2. Frame-wise BCE must fit the artificial boundaries. If these disagree
   systematically across examples, its optimum can average incompatible
   frame labels, producing uncertain probabilities. A thresholded metric
   can conceal that uncertainty. SDTW can adjust the alignment instead.
3. Early diffuse alignments mix several states in the modified targets.
   As predictions improve and gamma decreases, the alignments may sharpen
   and transitions become clearer. Collapse or persistent blurring remains
   possible; a low loss alone does not establish successful learning.


In [ ]:
def solution_unfolded_strong():

    # initialize loss
    bce_loss = loss_fn = torch.nn.BCELoss()

    # initialize model
    model_strong_unfolded = Transcription_CNN(
        num_classes=train_data.num_classes, activation=output_activation
    ).to(device)

    optimizer = torch.optim.Adam(model_strong_unfolded.parameters(), lr=0.001)
    train_losses, valid_losses, precisions, recalls, fmeasures = [], [], [], [], []
    training_start = perf_counter()

    for epoch in range(100):
        # train step
        model_strong_unfolded.train()
        total_loss = 0.0
        for (
            waveforms,
            strong_targets_frames,
            weak_targets_frames,
            strong_targets_onsets,
            weak_targets_onsets,
            target_lengths_frames,
            target_lengths_onsets,
        ) in train_loader:
            optimizer.zero_grad()
            predictions = model_strong_unfolded(waveforms)

            # compute SDTW loss between predictions and unaligned weak targets
            weak_targets_frames_unfolded = torch.stack(
                [
                    weak_targets_frames[
                        b,
                        np.linspace(
                            0,
                            target_lengths_frames[b],
                            500,
                            endpoint=False,
                            dtype=int,
                        ),
                        :,
                    ]
                    for b in range(weak_targets_frames.shape[0])
                ],
                dim=0,
            )

            loss = bce_loss(predictions, weak_targets_frames_unfolded)  

            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(waveforms)
        train_losses.append(total_loss / len(train_data))

        # validation step
        model_strong_unfolded.eval()
        total_loss = 0.0
        with torch.no_grad():
            p_ = []
            r_ = []
            f_ = []
            for (
                waveforms,
                strong_targets_frames,
                weak_targets_frames,
                strong_targets_onsets,
                weak_targets_onsets,
                target_lengths_frames,
                target_lengths_onsets,
            ) in valid_loader:
                predictions = model_strong_unfolded(waveforms)

                weak_targets_frames_unfolded = torch.stack(
                    [
                        weak_targets_frames[
                            b,
                            np.linspace(
                                0,
                                target_lengths_frames[b],
                                500,
                                endpoint=False,
                                dtype=int,
                            ),
                            :,
                        ]
                        for b in range(weak_targets_frames.shape[0])
                    ],
                    dim=0,
                )
                loss = bce_loss(predictions, weak_targets_frames_unfolded)
                total_loss += loss.item() * len(waveforms)

                # calculate precision, recall, and F1 score for each batch
                pred_thresh = (predictions > 0.5) * 1.0
                TP = (strong_targets_frames * pred_thresh).sum()
                FP = ((1 - strong_targets_frames) * pred_thresh).sum()
                FN = (strong_targets_frames * (1 - pred_thresh)).sum()
                P = TP / (TP + FP)
                R = TP / (TP + FN)
                F = 2 * P * R / (P + R)
                p_.append(P.item())
                r_.append(R.item())
                f_.append(F.item())

        valid_losses.append(total_loss / len(valid_data))

        # print stats every 10 epochs
        elapsed = perf_counter() - training_start
        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(
                (
                    "Epoch %3d | train loss %.5f | val loss %.4f | "
                    "val P %.4f | val R %.4f | val F1 %.4f | "
                    "elapsed time %.1f s"
                )
                % (
                    epoch + 1,
                    train_losses[-1],
                    valid_losses[-1],
                    np.mean(p_),
                    np.mean(r_),
                    np.mean(f_),
                    elapsed,
                )
            )
            precisions.append(np.mean(p_))
            recalls.append(np.mean(r_))
            fmeasures.append(np.mean(f_))

    # plot training and validation loss curves
    fig, ax = plt.subplots(1, 2, figsize=(6.2, 2.2))
    ax[0].plot(np.arange(1, len(train_losses) + 1), train_losses, label="train")
    ax[0].plot(
        np.arange(1, len(valid_losses) + 1), valid_losses, label="validation"
    )
    ax[0].set(
        xlabel="Epoch", ylabel="Mean loss", title="Training and validation loss"
    )
    ax[0].legend()
    ax[0].grid()

    ax[1].plot(np.arange(1, len(precisions) + 1), precisions, label="precision")
    ax[1].plot(np.arange(1, len(recalls) + 1), recalls, label="recall")
    ax[1].plot(np.arange(1, len(fmeasures) + 1), fmeasures, label="F1 score")
    ax[1].set(
        xlabel="Epoch", ylabel="Score", title="Precision, recall, and F1 score"
    )
    ax[1].legend()
    ax[1].grid()
    plt.show()

    ## plot predictions on a test waveform
    (
        test_waveform,
        test_strong_targets_frames,
        test_weak_targets_frames,
        _,
        _,
    ) = valid_data[0]
    pred_strong_unfolded = model_strong_unfolded(test_waveform[None])[0]


    fig, ax = plt.subplots(4, 1, figsize=(5.2, 5), squeeze=False)

    ax[0, 0].imshow(
        test_strong_targets_frames.numpy().T,
        aspect="auto",
        origin="lower",
        interpolation="None",
        cmap="gray_r",
        clim=[0, 1],
    )
    ax[1, 0].imshow(
        pred_strong_unfolded.detach().numpy().T,
        aspect="auto",
        origin="lower",
        interpolation="None",
        cmap="gray_r",
        clim=[0, 1],
    )

    # thresholded predictions
    pred_weak_thresh = (pred_strong_unfolded > 0.5) * 1.0
    TP = test_strong_targets_frames * pred_weak_thresh
    FP = (1 - test_strong_targets_frames) * pred_weak_thresh
    FN = test_strong_targets_frames * (1 - pred_weak_thresh)
    TN = (1 - test_strong_targets_frames) * (1 - pred_weak_thresh)
    pred_RGB_sdtw = np.zeros((500, len(class_names), 3))
    pred_RGB_sdtw[TP == 1] = [0, 0, 0]
    pred_RGB_sdtw[FP == 1] = [1, 0, 0]
    pred_RGB_sdtw[FN == 1] = [0, 1, 1]
    pred_RGB_sdtw[TN == 1] = [1, 1, 1]
    ax[2, 0].imshow(
        pred_RGB_sdtw.swapaxes(0, 1), aspect="auto", interpolation="None"
    )

    for d in range(len(class_names)):
        ax[3, 0].plot(np.arange(500), np.zeros(500) + d, color="grey", linewidth=2)
        ax[3, 0].plot(
            np.arange(500),
            pred_strong_unfolded.detach().numpy()[:, d] + d,
            color=f"C{d}",
            label=class_names[d],
            linewidth=1,
        )
        ax[3, 0].plot(
            np.arange(500),
            test_strong_targets_frames[:, d] + d,
            color=f"C{d}",
            linestyle="--",
        )
        ax[3, 0].grid()
    ax[3, 0].plot(np.arange(500), np.zeros(500) + d + 1, color="grey", linewidth=2)
    ax[3, 0].set(xlim=[-0.5, 499.5])


    ax[1, 0].set(ylabel="Probabilities")
    ax[2, 0].set(ylabel="thresh. pred.")
    ax[3, 0].set(ylabel="signals", xlabel="Frame index")

    ax[0, 0].set(title="Unfolded weak targets and SDTW", ylabel="Strong targets")
    ax[3, 0].set(xlabel="Frame index")

    red_patch = mpatches.Patch(color="red", label="FP")
    cyan_patch = mpatches.Patch(color="cyan", label="FN")
    black_patch = mpatches.Patch(color="black", label="TP")

    ax[2, 0].legend(
        handles=[black_patch, red_patch, cyan_patch], loc="upper right"
    )

    for a_ in ax.flatten():
        a_.set(yticks=np.arange(len(class_names)), yticklabels=class_names)

    fig.tight_layout()
    plt.show()

In [ ]:
RUN_STRONG_UNFOLDED_EXERCISE = False
if RUN_STRONG_UNFOLDED_EXERCISE:
    solution_unfolded_strong()

In [ ]:
def solution_unfolded_diagnostics():
    # initialize SDTW

    sdtw_loss = SDTW(
        gamma=10.0,
        step_sizes=[[1, 0], [1, 1], [0, 1]],
        cost_function="BCE",
        store_debug=True,
        global_step_weights=[1.0, 1.0, 1.0],
        normalization="N",
        backend="cpu_numba",
    )

    # initialize model
    model_sdtw_unfolded = Transcription_CNN(
        num_classes=train_data.num_classes, activation=output_activation
    ).to(device)

    optimizer = torch.optim.Adam(model_sdtw_unfolded.parameters(), lr=0.001)
    train_losses, valid_losses, precisions, recalls, fmeasures = (
        [],
        [],
        [],
        [],
        [],
    )
    training_start = perf_counter()

    for epoch in range(100):
        # train step
        model_sdtw_unfolded.train()
        total_loss = 0.0
        for (
            waveforms,
            strong_targets_frames,
            weak_targets_frames,
            strong_targets_onsets,
            weak_targets_onsets,
            target_lengths_frames,
            target_lengths_onsets,
        ) in train_loader:
            optimizer.zero_grad()
            predictions = model_sdtw_unfolded(waveforms)

            # compute SDTW loss between predictions and unaligned weak targets
            weak_targets_frames_unfolded = torch.stack(
                [
                    weak_targets_frames[
                        b,
                        np.linspace(
                            0,
                            target_lengths_frames[b],
                            500,
                            endpoint=False,
                            dtype=int,
                        ),
                        :,
                    ]
                    for b in range(weak_targets_frames.shape[0])
                ],
                dim=0,
            )

            loss = sdtw_loss(
                X=predictions, Y=weak_targets_frames_unfolded
            )  # , list_M=target_lengths_frames)

            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(waveforms)
        train_losses.append(total_loss / len(train_data))

        # validation step
        model_sdtw_unfolded.eval()
        total_loss = 0.0
        with torch.no_grad():
            p_ = []
            r_ = []
            f_ = []
            for (
                waveforms,
                strong_targets_frames,
                weak_targets_frames,
                strong_targets_onsets,
                weak_targets_onsets,
                target_lengths_frames,
                target_lengths_onsets,
            ) in valid_loader:
                predictions = model_sdtw_unfolded(waveforms)

                weak_targets_frames_unfolded = torch.stack(
                    [
                        weak_targets_frames[
                            b,
                            np.linspace(
                                0,
                                target_lengths_frames[b],
                                500,
                                endpoint=False,
                                dtype=int,
                            ),
                            :,
                        ]
                        for b in range(weak_targets_frames.shape[0])
                    ],
                    dim=0,
                )
                loss = sdtw_loss(
                    X=predictions, Y=weak_targets_frames_unfolded
                )  # , list_M=target_lengths_frames)

                total_loss += loss.item() * len(waveforms)

                # calculate precision, recall, and F1 score for each batch
                pred_thresh = (predictions > 0.5) * 1.0
                TP = (strong_targets_frames * pred_thresh).sum()
                FP = ((1 - strong_targets_frames) * pred_thresh).sum()
                FN = (strong_targets_frames * (1 - pred_thresh)).sum()
                P = TP / (TP + FP)
                R = TP / (TP + FN)
                F = 2 * P * R / (P + R)
                p_.append(P.item())
                r_.append(R.item())
                f_.append(F.item())

        valid_losses.append(total_loss / len(valid_data))

        # print stats every 10 epochs
        elapsed = perf_counter() - training_start
        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(
                (
                    "Epoch %3d | train loss %.5f | val loss %.4f | "
                    "val P %.4f | val R %.4f | val F1 %.4f | "
                    "elapsed time %.1f s"
                )
                % (
                    epoch + 1,
                    train_losses[-1],
                    valid_losses[-1],
                    np.mean(p_),
                    np.mean(r_),
                    np.mean(f_),
                    elapsed,
                )
            )
            precisions.append(np.mean(p_))
            recalls.append(np.mean(r_))
            fmeasures.append(np.mean(f_))

        if epoch == 0 or (epoch + 1) % 10 == 0:
            predictions_debug = model_sdtw_unfolded(waveforms[:4])
            debug_loss = sdtw_loss(
                X=predictions_debug, Y=weak_targets_frames_unfolded[:4]
            )
            # No optimizer step: backward only populates debug matrices.
            debug_loss.backward()
            fig, axes = plt.subplots(3, 4, figsize=(6.2, 3.6), layout="tight")
            for b in range(4):
                E = sdtw_loss.core.E_matrix[b].detach()
                H = sdtw_loss.core.H_matrix[b].detach()
                H_tilde = H / H.sum(dim=1, keepdim=True)
                Y_mod = H_tilde @ weak_targets_frames_unfolded[b]
                for row, values in enumerate(
                    [E, Y_mod, strong_targets_frames[b]]
                ):
                    axes[row, b].imshow(
                        values.numpy().T,
                        origin="lower",
                        aspect="auto",
                        cmap="gray_r",
                        interpolation="none",
                    )
            axes[0, 0].set_ylabel("Alignment E")
            axes[1, 0].set_ylabel("Modified targets")
            axes[2, 0].set_ylabel("Strong targets")
            plt.show()

        # do a gamma annealing every 20 epochs
        if (epoch + 1) % 20 == 0:
            sdtw_loss.gamma *= 0.5
            print("Gamma updated to %.2f" % sdtw_loss.gamma)

    # plot training and validation loss curves
    fig, ax = plt.subplots(1, 2, figsize=(6.2, 2.2))
    ax[0].plot(
        np.arange(1, len(train_losses) + 1), train_losses, label="train"
    )
    ax[0].plot(
        np.arange(1, len(valid_losses) + 1), valid_losses, label="validation"
    )
    ax[0].set(
        xlabel="Epoch",
        ylabel="Mean loss",
        title="Training and validation loss",
    )
    ax[0].legend()
    ax[0].grid()

    ax[1].plot(
        np.arange(1, len(precisions) + 1), precisions, label="precision"
    )
    ax[1].plot(np.arange(1, len(recalls) + 1), recalls, label="recall")
    ax[1].plot(np.arange(1, len(fmeasures) + 1), fmeasures, label="F1 score")
    ax[1].set(
        xlabel="Epoch", ylabel="Score", title="Precision, recall, and F1 score"
    )
    ax[1].legend()
    ax[1].grid()
    plt.show()

In [ ]:
RUN_UNFOLDED_EXERCISE = False
if RUN_UNFOLDED_EXERCISE:
    solution_unfolded_diagnostics()